In [1]:
import sys
!{sys.executable} -m pip install ipython-autotime
!{sys.executable} -m pip install xgboost
!{sys.executable} -m pip install torch
!{sys.executable} -m pip install optuna
!{sys.executable} -m pip install openpyxl

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [1]:
%load_ext autotime

import os
import random
import gc
from itertools import combinations
from collections import defaultdict
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import IterableDataset, DataLoader
import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, log_loss
import joblib
import zipfile
import ast

time: 6.47 s (started: 2026-05-26 08:06:27 +09:00)


### File Load

In [2]:
base_path = "C:\\Users\\woolz\\Downloads\\open\\"
file_names = [
    "train_xgb_42_10.parquet",
    "train_xgb_106_10.parquet",
    "train_xgb_1031_10.parquet",
    "test.parquet"
]

datasets = {}
for i, fname in enumerate(file_names, start=1):
    df = pd.read_parquet(base_path + fname)
    if "test" in fname:
        key = "test"
    else:
        key = f"train_{i:02d}"
    datasets[key] = df
    print(f"[Loaded] {key} | shape={df.shape}")

# ========================
# 3. 접근 예시
# ========================
train_01 = datasets["train_01"]
train_02 = datasets["train_02"]
train_03 = datasets["train_03"]
test     = datasets["test"]

[Loaded] train_01 | shape=(2245969, 119)
[Loaded] train_02 | shape=(2245969, 119)
[Loaded] train_03 | shape=(2245969, 119)
[Loaded] test | shape=(1527298, 119)
time: 38.5 s (started: 2026-05-26 08:06:43 +09:00)


### Data Processing

In [3]:
train_01.drop(columns=['l_feat_23', 'l_feat_20'], inplace=True)
train_02.drop(columns=['l_feat_23', 'l_feat_20'], inplace=True)
train_03.drop(columns=['l_feat_23', 'l_feat_20'], inplace=True)
test.drop(columns=['l_feat_23', 'l_feat_20'], inplace=True)

print("Train shape:", train_01.shape)
print("Train shape:", train_02.shape)
print("Train shape:", train_03.shape)
print("Test shape:", test.shape)

Train shape: (2245969, 117)
Train shape: (2245969, 117)
Train shape: (2245969, 117)
Test shape: (1527298, 117)
time: 1.96 s (started: 2026-05-26 08:07:29 +09:00)


In [4]:
def create_combination_features(df):

    base_cols = ['gender', 'age_group', 'inventory_id', 'day_of_week', 'hour']
    combo_features = {}

    for col_a, col_b in combinations(base_cols, 2): # combinations: base_cols 중 2개씩 뽑기
        combo_name = f'{col_a}_{col_b}'
        combo_features[combo_name] = (df[col_a].astype(str) + '_' + df[col_b].astype(str)).astype(object)

    combo_df = pd.DataFrame(combo_features, index=df.index) # index=df.index: 기존 df와 인덱스를 동일하게 맞추기
    df = pd.concat([df, combo_df], axis=1)

    print(f"✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})") # 총 10개의 feature 생성
    return df

train_01 = create_combination_features(train_01)
train_02 = create_combination_features(train_02)
train_03 = create_combination_features(train_03)
test     = create_combination_features(test)

✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 127)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 127)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 127)
✅ 범주형 기반 상호작용 파생변수 생성 완료 (총 컬럼 수: 127)
time: 15.7 s (started: 2026-05-26 08:07:35 +09:00)


In [5]:
df_click_prob = pd.read_excel('high_click_numbers.xlsx')

click_prob_map = dict(
    zip(df_click_prob['number'], df_click_prob['click_prob'])
)

pos_list = {370, 528, 68, 561, 144, 227, 417, 442, 186, 395}
neg_list = {154, 222, 84, 498, 434, 511, 216, 497, 309, 446}

def add_seq_features(df, name="dataset"):
    

    seq_len, avg_prob = [], []
    seq_neg, seq_pos = [], []
    pos_ratio, neg_ratio = [], []
    seq_margin, seq_balance = [], []
    max_prob, var_prob = [], []

    for s in df["seq"]:
        if isinstance(s, str) and s != "":
            arr = [int(x) for x in s.split(",") if x]

            n = len(arr)
            seq_len.append(n)

            probs = [
                click_prob_map[num]
                for num in arr
                if num in click_prob_map
            ]

            avg_prob.append(np.mean(probs) if probs else np.nan)
            max_prob.append(np.max(probs) if probs else np.nan)
            var_prob.append(np.var(probs, ddof=0) if len(probs) > 0 else np.nan)

            pos_cnt = sum(1 for x in arr if x in pos_list)
            neg_cnt = sum(1 for x in arr if x in neg_list)

            seq_pos.append(pos_cnt)
            seq_neg.append(neg_cnt)

            pos_ratio.append(pos_cnt / (n + 1))
            neg_ratio.append(neg_cnt / (n + 1))

            seq_margin.append(pos_cnt - neg_cnt)
            seq_balance.append((pos_cnt + 1) / (neg_cnt + 1))

        else:
            seq_len.append(0)
            avg_prob.append(np.nan)
            max_prob.append(np.nan)
            var_prob.append(np.nan)

            seq_neg.append(0)
            seq_pos.append(0)

            pos_ratio.append(0)
            neg_ratio.append(0)

            seq_margin.append(0)
            seq_balance.append(1)

    df["seq_len"] = seq_len
    df["avg_click_prob"] = avg_prob
    df["seq_neglogcount"] = seq_neg
    df["seq_poslogcount"] = seq_pos

    df["seq_pos_ratio"] = pos_ratio
    df["seq_neg_ratio"] = neg_ratio
    df["seq_margin"] = seq_margin
    df["seq_balance"] = seq_balance
    df["seq_max_click_prob"] = max_prob
    df["seq_prob_variance"] = var_prob

    print(f"✅ {name} seq 기반 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})")
    return df

train_01 = add_seq_features(train_01)
train_02 = add_seq_features(train_02)
train_03 = add_seq_features(train_03)
test = add_seq_features(test)

✅ dataset seq 기반 파생변수 생성 완료 (총 컬럼 수: 137)
✅ dataset seq 기반 파생변수 생성 완료 (총 컬럼 수: 137)
✅ dataset seq 기반 파생변수 생성 완료 (총 컬럼 수: 137)
✅ dataset seq 기반 파생변수 생성 완료 (총 컬럼 수: 137)
time: 39min 59s (started: 2026-05-26 08:07:56 +09:00)


In [6]:
full_train_df = pd.concat([train_01, train_02, train_03], ignore_index=True) # 기존 인덱스 버리고 다시 0부터 정렬

agg_targets = ['history_a_1','history_a_2','history_a_3', 'history_a_6','feat_d_4','l_feat_1','l_feat_2']

agg_stats = (
    full_train_df.groupby('inventory_id')[agg_targets]
    .agg(['mean','std'])
    .reset_index() # groupby 결과의 index(inventory_id)를 일반 column으로 변환
)

agg_stats.columns = ['inventory_id'] + [
    f'inventory_id_{col}_{stat}' for col, stat in agg_stats.columns[1:]
]

count_cols = ['age_group_inventory_id', 'inventory_id', 'inventory_id_hour']
count_maps = {col: full_train_df[col].value_counts().to_dict() for col in count_cols}

del full_train_df
gc.collect()


def add_features(df: pd.DataFrame, name: str = "dataset") -> pd.DataFrame:
    """groupby 통계량 + count encoding + 결측치 처리"""
    print(f"\n🚀 {name} 처리 시작...")

    df = df.merge(agg_stats, on='inventory_id', how='left')
    df.fillna(0, inplace=True)  # 숫자형 결측치 처리 (inplace=True: 원본 df 직접 수정)

    for col, cmap in count_maps.items():
        df[f"{col}_count"] = df[col].astype(str).map(cmap).fillna(0).astype(int)

    obj_cols = df.select_dtypes(include='object').columns
    df[obj_cols] = df[obj_cols].fillna("missing").astype(str)

    print(f"✅ {name}: seq 기반 파생변수 생성 완료 (총 컬럼 수: {df.shape[1]})")
    return df

train_01 = add_features(train_01)
train_02 = add_features(train_02)
train_03 = add_features(train_03)
test     = add_features(test)


🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 154)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 154)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 154)

🚀 dataset 처리 시작...
✅ dataset: seq 기반 파생변수 생성 완료 (총 컬럼 수: 154)
time: 1min 5s (started: 2026-05-26 08:48:24 +09:00)


### Modeling

In [7]:
## 튜닝코드

# -----------------------
# 클래스 균형 가중치
# -----------------------
def make_class_balanced_weights(y_true):
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    w_pos = 0.5 / (n_pos if n_pos > 0 else 1)
    w_neg = 0.5 / (n_neg if n_neg > 0 else 1)
    return np.where(y_true == 1, w_pos, w_neg)

# -----------------------
# 전처리
# -----------------------
BASE_EXCLUDE = {"clicked", "ID", "seq"}

def preprocess(df, exclude_features=None):
    exclude = BASE_EXCLUDE.copy()
    if exclude_features:
        exclude |= set(exclude_features)
    X = df.drop(columns=[c for c in exclude if c in df.columns], errors="ignore")

    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].astype("category").cat.codes.astype("int32")
    for col in X.select_dtypes(include=["int64"]).columns:
        X[col] = X[col].astype("int32")
    for col in X.select_dtypes(include=["float64"]).columns:
        X[col] = X[col].astype("float32")
    return X

# -----------------------
# Optuna 튜닝 함수
# -----------------------
def tune_xgb_with_optuna(X, y, seed, n_trials=5):
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=seed
    )

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)

    scale_pos_weight = float((y_train == 0).sum() / max(1, (y_train == 1).sum()))
    sampler = TPESampler(seed=seed)
    study = optuna.create_study(direction="maximize", sampler=sampler)

    def objective(trial):
        params = {
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "device" : 'cuda',
            "scale_pos_weight": scale_pos_weight,
            "eta": trial.suggest_float("eta", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 4, 12),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 200.0),
            "gamma": trial.suggest_float("gamma", 0.0, 5.0),
            "lambda": trial.suggest_float("lambda", 1e-8, 10.0, log=True),
            "alpha": trial.suggest_float("alpha", 1e-8, 10.0, log=True),
        }
        booster = xgb.train(
            params, dtrain,
            num_boost_round=1000,
            evals=[(dval, "valid")],
            early_stopping_rounds=30,
            verbose_eval=False
        )
        p_valid = booster.predict(dval, iteration_range=(0, booster.best_iteration + 1))
        ap = average_precision_score(y_val, p_valid)
        wll = log_loss(y_val, p_valid, sample_weight=make_class_balanced_weights(y_val), labels=[0, 1])
        final_score = 0.5 * ap + 0.5 * (1.0 / (1.0 + wll))
        return final_score

    study.optimize(objective, n_trials=n_trials)
    best_params = study.best_params
    best_params.update({
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "tree_method": "hist",
        "device" : "cuda",
        "scale_pos_weight": float((y == 0).sum() / max(1, (y == 1).sum()))
    })
    return best_params

# -----------------------
# 학습+예측 함수
# -----------------------
def train_and_predict(train_df, test_df, id_tag, seed, n_trials=50):
    print(f"\n=== START PIPELINE: {id_tag} | seed={seed} ===")

    random.seed(seed)
    np.random.seed(seed)

    y = train_df["clicked"].astype(int).values.ravel()
    X = preprocess(train_df)
    X_test = preprocess(test_df)
    test_ids = test_df["ID"].values if "ID" in test_df.columns else np.arange(len(test_df))

    # Optuna 탐색
    print(f"[{id_tag}] Optuna tuning ...")
    best_params = tune_xgb_with_optuna(X, y, seed=seed, n_trials=n_trials)
    print(f"[{id_tag}] Best params:", best_params)

    # 최종 학습
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dval = xgb.DMatrix(X_val, label=y_val)
    print(f"[{id_tag}] Training final model ...")
    model = xgb.train(
        best_params,
        dtrain,
        num_boost_round=1000,
        evals=[(dtrain, "train"), (dval, "valid")],
        early_stopping_rounds=50,
        verbose_eval=100
    )

    # 모델 저장
    model_path = os.path.join('C:\\Users\\woolz\\Downloads\\open\\', f"final_model_{id_tag}.pkl")
    joblib.dump(model, model_path)
    print(f"[{id_tag}] Model saved -> {model_path} | best_iter={model.best_iteration}")

    # 테스트 예측
    dtest = xgb.DMatrix(X_test)
    preds = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))
    sub = pd.DataFrame({"ID": test_ids, "clicked": preds})
    csv_path = os.path.join('C:\\Users\\woolz\\Downloads\\open\\', f"final_submission_{id_tag}.csv")
    sub.to_csv(csv_path, index=False)
    print(f"[{id_tag}] Submission saved -> {csv_path} | shape={sub.shape}")
    print(f"=== END PIPELINE: {id_tag} ===\n")
    return model_path, csv_path

# -----------------------
# 실행
# -----------------------
datasets = [
    {"df": train_01, "id": "train_01", "seed": 106},
    {"df": train_02, "id": "train_02", "seed": 1031},
    {"df": train_03, "id": "train_03", "seed": 42},
]

results = {}
for data in datasets:
    model_path, csv_path = train_and_predict(
        train_df=data["df"],
        test_df=test,
        id_tag=data["id"],
        seed=data["seed"],
        n_trials=50
    )
    results[data["id"]] = {"model": model_path, "submission": csv_path}

print("=== ALL DONE ===")
print(results)


=== START PIPELINE: train_01 | seed=106 ===
[train_01] Optuna tuning ...


[I 2026-05-26 08:50:00,685] A new study created in memory with name: no-name-ef23c248-4f78-42f9-87b1-84df2040634b
[I 2026-05-26 08:51:07,512] Trial 0 finished with value: 0.4564188345643359 and parameters: {'eta': 0.010259898367843918, 'max_depth': 12, 'subsample': 0.7798260137416737, 'colsample_bytree': 0.8440875086908464, 'min_child_weight': 170.55990367758548, 'gamma': 3.6940510569846796, 'lambda': 5.886868084847547, 'alpha': 0.5213830060758549}. Best is trial 0 with value: 0.4564188345643359.
[I 2026-05-26 08:51:51,462] Trial 1 finished with value: 0.43038616170326327 and parameters: {'eta': 0.12969245549678007, 'max_depth': 11, 'subsample': 0.7409115664353482, 'colsample_bytree': 0.6848127138204806, 'min_child_weight': 156.30843112731282, 'gamma': 4.4970562619261045, 'lambda': 2.6894091077470376e-08, 'alpha': 1.725942108719862e-07}. Best is trial 0 with value: 0.4564188345643359.
[I 2026-05-26 08:52:09,004] Trial 2 finished with value: 0.44710401687844475 and parameters: {'eta': 0

[train_01] Best params: {'eta': 0.017775092075747626, 'max_depth': 11, 'subsample': 0.9969572803849405, 'colsample_bytree': 0.6215639351655865, 'min_child_weight': 166.13846796819928, 'gamma': 0.5537318280851169, 'lambda': 2.880564773777019e-06, 'alpha': 4.005308284143297e-08, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_01] Training final model ...
[0]	train-logloss:0.69028	valid-logloss:0.69034
[100]	train-logloss:0.58597	valid-logloss:0.59184
[200]	train-logloss:0.56361	valid-logloss:0.57411
[300]	train-logloss:0.55273	valid-logloss:0.56610
[400]	train-logloss:0.54574	valid-logloss:0.56124
[500]	train-logloss:0.53982	valid-logloss:0.55724
[600]	train-logloss:0.53499	valid-logloss:0.55400
[700]	train-logloss:0.53000	valid-logloss:0.55071
[800]	train-logloss:0.52513	valid-logloss:0.54751
[900]	train-logloss:0.52095	valid-logloss:0.54477
[999]	train-logloss:0.51690	valid-logloss:0.54213
[train_01] Mo

[I 2026-05-26 09:25:33,392] A new study created in memory with name: no-name-610e5ca7-4f32-4672-9ccb-4e81685bc828
[I 2026-05-26 09:25:58,486] Trial 0 finished with value: 0.4496214074944703 and parameters: {'eta': 0.010030542235435747, 'max_depth': 6, 'subsample': 0.7299275357796907, 'colsample_bytree': 0.7672919056877435, 'min_child_weight': 154.42315013332836, 'gamma': 4.788695606033571, 'lambda': 5.838643100318619e-07, 'alpha': 0.0010984897811838177}. Best is trial 0 with value: 0.4496214074944703.
[I 2026-05-26 09:26:39,448] Trial 1 finished with value: 0.4184356639319501 and parameters: {'eta': 0.11978434297771987, 'max_depth': 10, 'subsample': 0.8028360964022752, 'colsample_bytree': 0.9033177780841783, 'min_child_weight': 17.555385408203744, 'gamma': 3.2365936437527756, 'lambda': 3.0923631972991695e-05, 'alpha': 0.1762792686327148}. Best is trial 0 with value: 0.4496214074944703.
[I 2026-05-26 09:26:56,380] Trial 2 finished with value: 0.45360851087779774 and parameters: {'eta': 

[train_02] Best params: {'eta': 0.013944956502259662, 'max_depth': 12, 'subsample': 0.737633082043145, 'colsample_bytree': 0.8748030301990103, 'min_child_weight': 159.75475138049936, 'gamma': 2.3639407806236834, 'lambda': 1.1341490418469808e-08, 'alpha': 0.007877535123525894, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_02] Training final model ...
[0]	train-logloss:0.69071	valid-logloss:0.69078
[100]	train-logloss:0.58741	valid-logloss:0.59396
[200]	train-logloss:0.56180	valid-logloss:0.57325
[300]	train-logloss:0.54908	valid-logloss:0.56401
[400]	train-logloss:0.54141	valid-logloss:0.55875
[500]	train-logloss:0.53507	valid-logloss:0.55448
[600]	train-logloss:0.52935	valid-logloss:0.55066
[700]	train-logloss:0.52389	valid-logloss:0.54707
[800]	train-logloss:0.51908	valid-logloss:0.54396
[900]	train-logloss:0.51409	valid-logloss:0.54069
[999]	train-logloss:0.50918	valid-logloss:0.53748
[train_02] Mod

[I 2026-05-26 09:59:12,769] A new study created in memory with name: no-name-b972c98d-9423-40e2-9993-4b5f156e061c
[I 2026-05-26 10:00:15,504] Trial 0 finished with value: 0.4467713233191961 and parameters: {'eta': 0.030710573677773714, 'max_depth': 12, 'subsample': 0.892797576724562, 'colsample_bytree': 0.8394633936788146, 'min_child_weight': 32.04770944804487, 'gamma': 0.7799726016810132, 'lambda': 3.3323645788192616e-08, 'alpha': 0.6245760287469893}. Best is trial 0 with value: 0.4467713233191961.
[I 2026-05-26 10:00:51,353] Trial 1 finished with value: 0.44835962451784056 and parameters: {'eta': 0.06054365855469246, 'max_depth': 10, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'min_child_weight': 166.65608551928392, 'gamma': 1.0616955533913808, 'lambda': 4.329370014459266e-07, 'alpha': 4.4734294104626844e-07}. Best is trial 1 with value: 0.44835962451784056.
[I 2026-05-26 10:01:18,919] Trial 2 finished with value: 0.4548308121242962 and parameters: {'eta':

[train_03] Best params: {'eta': 0.015203207612990325, 'max_depth': 12, 'subsample': 0.8873645429507125, 'colsample_bytree': 0.8910048511400469, 'min_child_weight': 198.26416203257625, 'gamma': 2.5622334242299454, 'lambda': 9.957610803208073e-07, 'alpha': 3.7134270187779125e-08, 'objective': 'binary:logistic', 'eval_metric': 'logloss', 'tree_method': 'hist', 'device': 'cuda', 'scale_pos_weight': 10.0}
[train_03] Training final model ...
[0]	train-logloss:0.69054	valid-logloss:0.69062
[100]	train-logloss:0.58604	valid-logloss:0.59280
[200]	train-logloss:0.56142	valid-logloss:0.57330
[300]	train-logloss:0.54871	valid-logloss:0.56415
[400]	train-logloss:0.54074	valid-logloss:0.55866
[500]	train-logloss:0.53488	valid-logloss:0.55471
[600]	train-logloss:0.52912	valid-logloss:0.55088
[700]	train-logloss:0.52375	valid-logloss:0.54733
[800]	train-logloss:0.51883	valid-logloss:0.54413
[900]	train-logloss:0.51411	valid-logloss:0.54106
[999]	train-logloss:0.50931	valid-logloss:0.53794
[train_03] M

In [8]:
# ✅ 1) 파일 불러오기 (Drive에서 불러오기)
df1 = pd.read_csv(os.path.join("final_submission_train_01.csv"))
df2 = pd.read_csv(os.path.join("final_submission_train_02.csv"))
df3 = pd.read_csv(os.path.join("final_submission_train_03.csv"))

# ✅ 2) ID 기준 병합
merged = df1.merge(df2, on="ID", suffixes=("_1", "_2"))
merged = merged.merge(df3, on="ID")
merged.rename(columns={"clicked": "clicked_3"}, inplace=True)

# ✅ 3) Soft Voting (평균)
merged["clicked"] = (
    merged["clicked_1"] + merged["clicked_2"] + merged["clicked_3"]
) / 3

# ✅ 4) ID와 최종 clicked만 추출
final = merged[["ID", "clicked"]]

# ✅ 5) 저장 (→ /content 폴더에 저장)
output_path = os.path.join('C:\\Users\\woolz\\Downloads\\open\\', "final_xgb.csv")
final.to_csv(output_path, index=False)

print(f"🎯 Soft Voting 결과 저장 완료!\n파일 경로: {output_path}")
print(final.head())

🎯 Soft Voting 결과 저장 완료!
파일 경로: C:\Users\woolz\Downloads\open\final_xgb.csv
             ID   clicked
0  TEST_0000000  0.305390
1  TEST_0000001  0.304599
2  TEST_0000002  0.345697
3  TEST_0000003  0.440419
4  TEST_0000004  0.287474
time: 4.57 s (started: 2026-05-26 11:30:58 +09:00)
